# Fine-tune with Unsloth

In [1]:
pip install -r requirements.txt

  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 295.2 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 351.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 326.4 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 372.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 382.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 259.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 311.0 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 367.2 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 155.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 382.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 372.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━

In [1]:
from typing import Dict, Any
from unsloth import FastLanguageModel, is_bfloat16_supported
from datasets import load_dataset
import torch
from transformers import TrainingArguments
from trl import SFTTrainer

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
dataset_path = "training_pairs_new.json"
text_field = "text"
output_dir = "outputs/lfm25-lora-qwen"
max_seq_length = 2048
load_in_4bit = False
batch_size = 8
grad_accum = 8
learning_rate = 1.5e-4
max_steps = 7000
warmup_steps = 210
save_steps = 500
logging_steps = 10
seed = 3407
test_size = 0.05
resume_from_checkpoint = None  # Set to True for latest, or a checkpoint path like "outputs/lfm25-unsloth-lora/checkpoint-1800"

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not found. This notebook requires CUDA for Unsloth fine-tuning.")

def print_vram():
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"VRAM: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")

use_bf16 = is_bfloat16_supported()

In [8]:
pip install flash-attn-4

ERROR: Ignored the following yanked versions: 0.0.1
ERROR: Could not find a version that satisfies the requirement flash-attn-4 (from versions: 4.0.0b3, 4.0.0b4)
ERROR: No matching distribution found for flash-attn-4
Note: you may need to restart the kernel to use updated packages.


In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen3.5-4B-Base",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=load_in_4bit,
    attn_implementation="flash_attention_2",
)

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=seed,
    use_rslora=False,
    loftq_config=None,
)

==((====))==  Unsloth 2026.3.4: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA B200. Num GPUs = 1. Max memory: 178.351 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 10.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


ImportError: FlashAttention2 has been toggled on, but it cannot be used due to the following error: the package flash_attn seems to be not installed. Please refer to the documentation of https://huggingface.co/docs/transformers/perf_infer_gpu_one#flashattention-2 to install Flash Attention 2.

In [5]:
dataset = load_dataset("json", data_files=dataset_path, split="train")

def messages_to_text(messages: Any) -> str:
    if not isinstance(messages, list):
        return str(messages)
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        ).strip()
    except Exception:
        parts = []
        for item in messages:
            if not isinstance(item, dict):
                continue
            role = str(item.get("role", "user"))
            content = str(item.get("content", "")).strip()
            if content:
                parts.append(f"{role}: {content}")
        return "\n".join(parts).strip()

if text_field in dataset.column_names:
    def ensure_text(example: Dict[str, Any]) -> Dict[str, str]:
        value = example[text_field]
        return {"text": value if isinstance(value, str) else str(value)}
elif "messages" in dataset.column_names:
    print("'text' field not found; using 'messages' column and chat template formatting.")
    def ensure_text(example: Dict[str, Any]) -> Dict[str, str]:
        return {"text": messages_to_text(example["messages"])}
else:
    raise ValueError(f"Field '{text_field}' not found in dataset columns: {dataset.column_names}")

train_dataset = dataset.map(ensure_text, remove_columns=dataset.column_names)
train_dataset = train_dataset.filter(lambda x: bool(x["text"].strip()))
split_dataset = train_dataset.train_test_split(test_size=test_size, seed=seed)

'text' field not found; using 'messages' column and chat template formatting.


In [6]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=True,
    args=TrainingArguments(
        eval_strategy="steps",
        eval_steps=save_steps,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        warmup_steps=warmup_steps,
        max_steps=max_steps,
        learning_rate=learning_rate,
        fp16=not use_bf16,
        bf16=use_bf16,
        logging_steps=logging_steps,
        optim="adamw_torch",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=seed,
        output_dir=output_dir,
        save_steps=save_steps,
        report_to="none",
    ),
)

print_vram()
trainer.train(resume_from_checkpoint=resume_from_checkpoint)
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248044}.


VRAM: 8.78GB allocated, 8.79GB reserved


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 80,885 | Num Epochs = 6 | Total steps = 7,000
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 8 x 1) = 64
 "-____-"     Trainable parameters = 84,934,656 of 4,624,200,192 (1.84% trained)


Step,Training Loss,Validation Loss


KeyboardInterrupt: 